# Environmental Footprints of Housing
This code calculates environmental footprints of housing under a given set of energy use, energy technology, housing size of different building types, and transport use in relation to location and income.

For the analysis of current housing in NL the following datasets are needed:
- Set of assumptions: CBS microdata output
- Material use per building types
- Material use per solar PV capacity
- Material use per heat pump
- District heat source distribution
- Fuel use per pkm
- Infrastructural material use per pkm

In [2]:
# data processing modules
import pandas as pd
import numpy as np

# data visualisation modules
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import seaborn as sns

## Foreground System: Calculate and Harmonize Direct Material, Fuel & Electricity Use

### Set of Assumptions
- configure assumptions
- import the CBS file

In [6]:
# Configurations
scenario = 'current'
material_buildings_source = 'Oorschot'
material_solarpv_source = 'ecoinvent'
material_heatpump_source = 'ecoinvent'
material_pkminfra_source = None
fuel_pkm_source = None
fuel_districtheating_source = None

# File import
if scenario == "current":
    use = pd.read_excel(
        "data_input/output/income typology construction density distance privacy 2019.xlsx",
        index_col=[0,1,2,3,4],
        header=[0],
    )

In [11]:
use

inc_decile  \
inc_lab             typology_lab              const_lab density_lab distance_train_lab               
decile 1 (poorest)  detached (vrijstaand)     pre 1945  very urban  nearest quintile             1   
                                                                    2nd quintile                 1   
                                                                    3nd quintile                 1   
                                                                    4nd quintile                 1   
                                                                    furthest quintile            1   
...                                                                                            ...   
decile 10 (richest) multiple fam (meergezins) 2006-2022 rural       nearest quintile            10   
                                                                    2nd quintile                10   
                                                                    3nd quintile                10   
                                                                    4nd quintile                10   
                                                                    furthest quintile           10   

                                                                                        typology4_2019  \
inc_lab             typology_lab              const_lab density_lab distance_train_lab                   
decile 1 (poorest)  detached (vrijstaand)     pre 1945  very urban  nearest quintile                 1   
                                                                    2nd quintile                     1   
                                                                    3nd quintile                     1   
                                                                    4nd quintile                     1   
                                                                    furthest quintile                1   
...                                                                                                ...   
decile 10 (richest) multiple fam (meergezins) 2006-2022 rural       nearest quintile                 4   
                                                                    2nd quintile                     4   
                                                                    3nd quintile                     4   
                                                                    4nd quintile                     4   
                                                                    furthest quintile                4   

                                                                                        const_cat2019  \
inc_lab             typology_lab              const_lab density_lab distance_train_lab                  
decile 1 (poorest)  detached (vrijstaand)     pre 1945  very urban  nearest quintile                1   
                                                                    2nd quintile                    1   
                                                                    3nd quintile                    1   
                                                                    4nd quintile                    1   
                                                                    furthest quintile               1   
...                                                                                               ...   
decile 10 (richest) multiple fam (meergezins) 2006-2022 rural       nearest quintile                6   
                                                                    2nd quintile                    6   
                                                                    3nd quintile                    6   
                                                                    4nd quintile                    6   
                                                                    furthest quintile               6   

                

In [10]:
use['size2019pp_w1']

inc_lab              typology_lab               const_lab  density_lab  distance_train_lab
decile 1 (poorest)   detached (vrijstaand)      pre 1945   very urban   nearest quintile      105.062950
                                                                        2nd quintile          109.136978
                                                                        3nd quintile           97.551064
                                                                        4nd quintile          129.263885
                                                                        furthest quintile            NaN
                                                                                                 ...    
decile 10 (richest)  multiple fam (meergezins)  2006-2022  rural        nearest quintile       72.181580
                                                                        2nd quintile           92.352272
                                                                     

### Building Construction
- translate floor space to BONSAI material categories
- include material use for maintenance
- normalise impact to annual using building lifetime

In [ ]:
# Configurations
mi_buildings_source = "Oorschot"  # Oorschot or Yang
# Load data
if mi_buildings_source == "Oorschot":
    mi_buildings = pd.read_excel(
        "data_input/MI_Buildings_Oorschot2023.xlsx",
        sheet_name="MI_2018",
        index_col=[0, 1],
        header=[0, 1],
    ).droplevel(1, axis=1)
    mi_buildings.index.names = ["Building type", "Cohort"]
    mi_buildings.columns.name = "Material"

    mi_buildings_2018 = pd.read_excel(
        "data_input/MI_Buildings_Oorschot2023.xlsx",
        sheet_name="MI_2018-2050",
        index_col=[0],
        header=[0, 1],
        skipfooter=20,
    ).droplevel(1, axis=1)
    mi_buildings_2018["Cohort"] = "2018"
    mi_buildings_2018.reset_index(inplace=True)
    mi_buildings_2018["index"] = [i.lower() for i in mi_buildings_2018["index"]]
    mi_buildings_2018.set_index(["index", "Cohort"], inplace=True)
    mi_buildings_2018.index.names = ["Building type", "Cohort"]
    mi_buildings_2018.columns.name = "Material"

    mi_buildings_insulation_share = pd.read_excel(
        "data_input/MI_Buildings_Oorschot2023.xlsx",
        sheet_name="Share_of_insulation_materials",
        index_col=[0, 1],
        header=[0],
        skipfooter=3,
    ).drop("Unnamed: 8", axis=1)
    for i in mi_buildings_insulation_share.columns:
        mi_buildings[i] = (
            mi_buildings["Insulation"] * mi_buildings_insulation_share[i].iloc[0]
        )
        mi_buildings_2018[i] = (
            mi_buildings_2018["Insulation"] * mi_buildings_insulation_share[i].iloc[1]
        )

    mi_buildings = (
        pd.concat([mi_buildings, mi_buildings_2018], axis=1)
        .T.groupby(level=0)
        .sum()
        .T.sort_index()
        .replace(0, np.nan)
        .dropna(how="all", axis=1)
        .drop("Insulation", axis=1)
    )

# matching material names
replacement_map = {'Bitumen':'Bitumen', #soon
                    'Brick':'Ceramics', #BRIK,
                    'Clay brick':'Ceramics', #BRIK
                    'Ceramics':'Ceramics', #BRIK (bricks, tiles, construction products)
                    'Sand-lime brick':'Ceramics', #BRIK
                    'Sand':'Sand', #SDCL
                    'Concrete':'Concrete', #CMNT (only lime,cement and plast, no mix)
                    'Mortal':'Mortar', #CMNT
                    'Gypsum':'Mortar', #CMNT (plaster of gypsum)
                    'Aluminum':'Aluminium', #ALUM
                    'Copper':'Copper', #COPP
                    'Other metals':'Other metals', #ONFM
                    'Electronics':'Electronics', #ELMA
                    'Glass':'Glass', #GLAS 
                    'Glass wool':'Glass', #GLAS (includes explicitely glass wool)
                    'Stone wool':'Mineral wool', #ONMM (includes explicitely mineral wool)
                    'Iron & steel':'Steel', #STEL or FABM
                    'Steel':'Steel', #STEL or FABM
                    'Other biobased materials':'Wood and straw', #WOOD
                    'Wood':'Wood and straw', #WOOD
                    'Other construction materials':'Other construction material', #ONMM, could also be STON, SDCL or CMNT
                    'Other':'Adhesives and paint', #could be CHEM, PLAS, PAPE, ELMA
                    'Paper':'Paper', #PAPE 
                    'Expanded polystyrene':'Plastic', #RUBP 
                    'Extruded polystyrene':'Plastic', #RUBP
                    'Plastic':'Plastic', #RUBP
                    'Plastics':'Plastic', #RUBP
                    'Polyurethanes':'Plastic', #RUBP
                    'Adhesives & paint':'Adhesives and paint', #CHEM (other chemical products, for glues) or PLAS (basic plastics) 
                    'Stone':'Stone', #STON
                    }

mi_buildings = mi_buildings.stack().reset_index()
mi_buildings['Material_group'] = mi_buildings['Material'].map(replacement_map)
mi_buildings.replace('<1945','1944', inplace=True)
mi_buildings = mi_buildings.drop('Material', axis=1).groupby(['Building type', 'Cohort', 'Material_group']).sum().unstack().droplevel(0, axis=1)

#selecting only residential buildings
mi_buildings.drop(['industry', 'office', 'services', 'shop'], axis=0, level=0, inplace=True)

In [ ]:
m_plot = mi_buildings.copy()
m_plot.index = [str(i) + ' ' + str(j) for i, j in m_plot.index]
m_plot = m_plot.stack().reset_index()
m_plot.columns = ['building', 'material', 'value']
fig = px.bar(m_plot, x='value', y='building', color='material', labels={'value':'material intensity per floor area [kg/m2]'}, template='simple_white', 
             color_discrete_sequence= px.colors.qualitative.Prism) #antique, vivid, prism, T10
fig.update_layout(width=1100, height=500, yaxis = dict(autorange="reversed"))    
fig.write_image('visualisations/mi_buildings'+mi_buildings_source+'.svg')
fig.show()

In [ ]:
# Configurations
mi_buildings_source = "Yang"  # Oorschot or Yang
# Load data
if mi_buildings_source == "Yang":
    mi_buildings = pd.read_excel(
        "data_input/MI_Buildings_Yang2022.xlsx",
        sheet_name='0_MaterialIntensity',
        index_col=[0, 1],
        header=[0, 1],
    ).droplevel(0, axis=0).droplevel(1, axis=1)
    mi_buildings.index.names = ["Building type"]
    mi_buildings.columns.name = "Material"

    # matching material names
    replacement_map = {'bitumen adhesive compound, hot':'Bitumen', #soon
                        'clay brick':'Ceramics', #BRIK
                        'ceramic tile':'Ceramics', #BRIK (bricks, tiles, construction products)
                        'sand':'Sand', #SDCL
                        'concrete, normal':'Concrete', #CMNT (only lime,cement and plast, no mix)
                        'fibre-reinforced concrete':'Concrete', #CMNT (only lime,cement and plast, no mix)
                        'cement mortar':'Mortar', #CMNT
                        'gypsum, mineral':'Mortar', #CMNT (plaster of gypsum)
                        'aluminium, wrought alloy':'Aluminium', #ALUM ##
                        'copper':'Copper', #COPP
                        'zinc':'Zinc', #LZNC
                        'flat glass, coated':'Glass', #GLAS 
                        'glass wool mat':'Glass', #GLAS (includes explicitely glass wool)
                        'cast iron':'Steel', #STEL or FABM
                        'wire drawing, steel':'Steel', #STEL or FABM
                        'plywood, for indoor use':'Wood and straw', #WOOD
                        'sawnwood, hardwood, raw':'Wood and straw', #WOOD
                        'sawnwood, softwood, raw':'Wood and straw', #WOOD
                        'fibreboard, soft':'Wood and straw', #WOOD
                        'polystyrene, expandable':'Plastic', #RUBP 
                        'polystyrene, extruded':'Plastic', #RUBP
                        'extrusion, plastic pipes':'Plastic', #RUBP
                        'polyurethane, flexible foam':'Plastic', #RUBP
                        'gravel, crushed':'Stone', #STON
                        'argon, liquid':'Argon', #CHEM
                        }

    mi_buildings = mi_buildings.stack().reset_index()
    mi_buildings['Material_group'] = mi_buildings['Material'].map(replacement_map)
    mi_buildings = mi_buildings.drop('Material', axis=1).groupby(['Building type', 'Material_group']).sum().unstack().droplevel(0, axis=1)

    nzeb = mi_buildings.loc[(mi_buildings.index.str.contains('_NZEB')) & ~(mi_buildings.index.str.contains('new'))]
    nzeb.index = [i[:-5] for i in nzeb.index]
    nzeb = nzeb + mi_buildings.loc[~(mi_buildings.index.str.contains('_NZEB')) & ~(mi_buildings.index.str.contains('new'))]
    nzeb.index = [i+'_NZEB' for i in nzeb.index]
    mi_buildings.update(nzeb)

In [ ]:
m_plot = mi_buildings[~mi_buildings.duplicated()]
m_plot = m_plot.stack().reset_index()
m_plot.columns = ['building', 'material', 'value']
fig = px.bar(m_plot, x='value', y='building', color='material', labels={'value':'material intensity per floor area [kg/m2]', 'sector':'Input sector'}, template='simple_white', 
             color_discrete_sequence= px.colors.qualitative.Prism) #antique, vivid, prism, T10
fig.update_layout(width=1100, height=700, yaxis = dict(autorange="reversed"))    
fig.write_image('visualisations/mi_buildings'+mi_buildings_source+'.svg')
fig.show()

### Building Use Phase
- translate district heating to BONSAI fuel categories
- translate solar PV and heat pumps to BONSAI material categories
- harmonise labels for electricity and gas use with BONSAI categories

### Mobility
- translate pkm to BONSAI fuel categories
- translate pkm to infrastructure demand to BONSAI material categories
- normalise cars to annual cars using car lifetime
- harmonise labels for cars with BONSAI categories

## Background System: Environmental Footprints

### Environmental impact factors
- import BONSAI multipliers

In [ ]:
prod_codes = {'M_SDCL':'Mining and quarrying, non-energy producing products_Sand and clay', #sand, clay
              'M_FABM':'Fabricated metal products_Steel', #iron, steel #ambiguous category: could also be classified as A_STEL production of basic iron and steel
              'M_ONFM':'Basic metals_Other metals', #other non-ferrous metal products
              'M_WOOD':'Wood and products of wood and cork_Timber', #wood, straw
              'M_LZTP':'Basic metals_Lead', #lead, zink, tin products
              'M_ALUM':'Basic metals_Aluminium', #aluminium products
              'M_COPP':'Basic metals_Copper', #copper products
              'M_GLAS':'Other non-metallic mineral products_Glass', #glass
              'M_RUBP':'Rubber and plastic products_Plastics', #rubber, plastic
              'M_STON':'Other non-metallic mineral products_Stone', #stone
              'M_BRIK':'Other non-metallic mineral products_Bricks', #bricks, tiles, baked clay construction products
              'M_CMNT':'Other non-metallic mineral products_Mortar',#cement, lime, plaster
              }

mult_bonsai = pd.read_csv('external_data/240605_BONSAI_ghg_for_all_iluc.csv', index_col=[0,1,2,3], header=[0], skipfooter=9)
mult_bonsai = mult_bonsai[mult_bonsai.index.get_level_values(0).isin(prod_codes.keys())] # per tonne
mult_bonsai = mult_bonsai.droplevel([1,2,3]).reset_index()
mult_bonsai['material'] = mult_bonsai['Exio prod code'].map(prod_codes)
mult_bonsai = mult_bonsai.drop('Exio prod code', axis=1).set_index('material')
concrete_cement_pc = 0.35 #based on M20 and M25 concrete grade
mult_bonsai.loc['Other non-metallic mineral products_Concrete'] = mult_bonsai.loc[['Other non-metallic mineral products_Mortar']].squeeze()*concrete_cement_pc + mult_bonsai.loc[['Mining and quarrying, non-energy producing products_Sand and clay']].squeeze()*(1-concrete_cement_pc)
asphalt_cement_pc = 0.25 #based on M10 and M15 concrete grade
mult_bonsai.loc['Other non-metallic mineral products_Asphalt concrete'] = mult_bonsai.loc[['Other non-metallic mineral products_Mortar']].squeeze()* asphalt_cement_pc + mult_bonsai.loc[['Mining and quarrying, non-energy producing products_Sand and clay']].squeeze()*(1-asphalt_cement_pc)

mult_bonsai.index = pd.MultiIndex.from_tuples([tuple(i.split('_')) for i in mult_bonsai.index], names=['Material_group', 'Material'])
mult_bonsai = pd.concat([mult_bonsai, (mult_bonsai.xs('Basic metals', axis=0, level=0)).assign(Material_group='Fabricated metal products').set_index('Material_group', append=True).reorder_levels(['Material_group', 'Material'])], axis=0)

# CODE ARCHIVE

In [ ]:
# Configurations
ei_heating_source = "TABULA"  # Oorschot or Yang
# Load data
if ei_heating_source == "TABULA":
    ei_heating = pd.read_excel(
        "data_input/EI_Heating_HotWater_TABULA2017.xlsx",
        sheet_name="0_EnergyDemandIntensity",
        index_col=[0],
        header=[0],
    ).drop('Default Heating & Hot Water', axis=1)

In [ ]:
m_plot = ei_heating.copy()

m_plot = m_plot.stack().reset_index()
m_plot.columns = ['building', 'function', 'value']
fig = px.bar(m_plot, x='value', y='building', color='function', labels={'value':'energy intensity per floor area [kWh/m2]'}, template='simple_white', 
             color_discrete_sequence= px.colors.qualitative.Prism) #antique, vivid, prism, T10
fig.update_layout(width=500, height=1200, yaxis = dict(autorange="reversed"))    
fig.write_image('visualisations/ei_heating'+ei_heating_source+'.svg')
fig.show()